# LAB 1 — Databricks Fundamentals & DEV Setup (Week 1)

## Importing and loading data from REST API

In [0]:
import requests

catalog = "dbr_dev"
schema = "jmizera"
volume = "lab1_volume"
file_name = "co2.csv"

path = f"/Volumes/{catalog}/{schema}/{volume}/{file_name}"
url = "https://owid-public.owid.io/data/co2/owid-co2-data.csv"

In [0]:
response = requests.get(url)
if response.status_code == 200:
    with open(path, "wb") as f:
        f.write(response.content)
else:
    print("Failed to download data. ")

In [0]:
co2_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path)
display(co2_df.limit(20))

## Loading data into a Delta table, saving to schema

In [0]:
table_name = "co2"
full_table_name = f"{catalog}.{schema}.{table_name}"

co2_df.write.format("delta").mode("overwrite").saveAsTable(full_table_name)

## Basic Spark operations

In [0]:
from pyspark.sql.functions import col

recent_co2_df = co2_df.select("country", "year", "co2", "population") \
    .filter((col("country") == "Poland") & (col("year") >= 2000))

display(recent_co2_df)

In [0]:
from pyspark.sql.functions import sum, desc

country_emission_df = co2_df.filter(col("year") >= 2020) \
    .groupBy("country") \
    .agg(sum("co2").alias("total_co2")) \
    .orderBy(desc("total_co2"))

display(country_emission_df.limit(20))

In [0]:
from pyspark.sql.functions import round, try_divide

gas_co2_ratio = co2_df.withColumn("gas_co2_ratio", round(try_divide(col("gas_co2"), col("co2")), 4))

display(gas_co2_ratio)

In [0]:
energy_source_df = co2_df.filter((col("year") >= 2000)) \
    .groupBy("country") \
    .agg(
        round(sum("coal_co2"), 2).alias("total_coal_co2"),
        round(sum("oil_co2"), 2).alias("total_oil_co2"),
        round(sum("gas_co2"), 2).alias("total_gas_co2")
    )

display(energy_source_df)

In [0]:
from pyspark.sql import Row

# Sample region dataframe for join operation
regions = [
    Row(country="Poland", region="Europe"),
    Row(country="Germany", region="Europe"),
    Row(country="France", region="Europe"),
    Row(country="Canada", region="North America"),
    Row(country="United States", region="North America"),
    Row(country="China", region="Asia"),
    Row(country="India", region="Asia"),
    Row(country="Japan", region="Asia")
]
regions_df = spark.createDataFrame(regions)

display(regions_df)

joined_df = co2_df.join(regions_df, on="country", how="inner")

display(joined_df)

new_table_name = "result"
full_table_name = f"{catalog}.{schema}.{new_table_name}"

joined_df.write.format("delta").mode("overwrite").saveAsTable(full_table_name)